# Script Outline



## Prepare Workspace

#### Import Packages

Need to check for invalid responses = NA, -1, 0?

Can also "Expand" the PUMS data by the "PWGTP" value per row (for example, if the PWGTP = 177, there are 177 people like that, so can expand to 177 rows)

Hispanic race/ethnicity is classified differently from other races (need another table)

Defined categories by variable change by year sometimes

In [ ]:
# General
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter

# Geographic
import geopandas as gpd
from census import Census
from us import states
import censusdata as acs

#### File paths

In [ ]:
# Define user
user = os.getlogin()

# Working directories
path_sp  = os.path.join('C:\\Users', 'jfontes', 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_git = os.path.join('C:\\Users', 'jfontes', 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

# Set file paths
path_config = os.path.join(path_git, 'Pipeline', 'Python Code', 'Census', 'aa_config')
path_out    = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')

#### User Defined Functions/Objects

In [ ]:
## User defined functions
exec(open(os.path.join(path_config, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

## Prepare Inputs for Importing ACS Data

#### Import ACS tables/variables mapping and FIPS mapping

In [ ]:
# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'Pipeline Configuration File.xlsx'), sheet_name = 'Parameters')

# Set parameters for querying ACS data
estimate    = df_params['estimate' ].values[0]
sample_type = df_params['samples'  ].values[0]
geography   = df_params['geography'].values[0]

inputs = sample_type + '_' + geography
proportions = df_params['proportions'].values[0]

In [ ]:
## Import Variable Mapping
df_vars   = pd.read_excel(os.path.join(path_config, 'Pipeline Configuration File.xlsx'), sheet_name = sample_type)
df_inputs = pd.read_excel(os.path.join(path_config, 'Pipeline Configuration File.xlsx'), sheet_name = inputs)

# Organize inputs into run
indicator_name = df_inputs['indicator_name'].values[0]
year_start = int(df_inputs['year_start'].values[0])
year_end   = int(df_inputs['year_end'  ].values[0])
sp_folder_out = df_inputs['sp_folder'].values[0]

# Subset variables
df_vars = df_vars[df_vars['Indicator Name'] == indicator_name]
df_vars = df_vars[df_vars['Include'] == 'Yes']

# Set variables to import
record_type = df_inputs['record_type'].values[0]
list_vars = ['PUMA'] + df_vars['ID'].to_list()
variables = ','.join(list_vars)
if record_type == 'P': 
    variables = ','.join([variables, 'PWGTP'])
if record_type == 'H': 
    variables = ','.join([variables,  'WGTP'])

# Set years
years_to_import = list(range(year_start, year_end+1))



## Import County FIPS mapping
df_fips = pd.read_excel(os.path.join(path_config, 'Pipeline Configuration File.xlsx')
                        , sheet_name = 'FIPSmap'
                        , dtype = {'State FIPS': object, 'County FIPS': object})
df_fips = df_fips[df_fips['State'].isin(df_inputs['states'].values)] 
states = df_fips['State FIPS'].unique()

# view
print(record_type)
print(states)
print(variables)
print(indicator_name)
print(year_start)
print(year_end)
df_vars.head()

## Import Data

#### Create Census Tracts Table

In [ ]:
# initialize empty list to store data frames
list_df_acs = []

# pull all years into one table (takes 2-3 minutes per year)
for state in states:
    print(state)
    for year in tqdm(years_to_import):
        try:
            list_df_acs.append(
                acs5_pums(api_Key       = api_key
                          , variables   = variables
                          , year        = year
                          , state       = state
                          , record_type = record_type)
                        )
        except:
            pass

# combine all years
df_acs_raw = pd.concat(list_df_acs)

In [ ]:
# Check years and counties
print(df_acs_raw['Year'].unique())
# assert df_acs_raw['Year'].unique().tolist() == years_to_import
# assert df_acs_raw['msa' ].unique().tolist() == msa_to_import


# view raw data
pd.set_option('display.max_columns', None)
print(df_acs_raw.shape)
print(df_acs_raw.Year.unique())
df_acs_raw.head(3)

## Data Cleaning

In [ ]:
df_acs = df_acs_raw.copy()
df_acs['PWGTP'] = df_acs['PWGTP'].astype(int)
df_acs = df_acs.drop(['RT'], axis = 1)
cols = ['state', 'PUMA', 'Year'] + list(df_acs.columns[1:-2])
df_acs = df_acs[cols]

df_acs = df_acs.groupby(list(df_acs.columns[:-1]), as_index = False, sort = False)['PWGTP'].sum()

df_acs = df_acs.sort_values(list(df_acs.columns[:-1]))

# view
df_acs.head()

#### Export

In [ ]:
# Set output name
name_output_long = ['ACS5 ', indicator_name, ' Long MSA.xlsx']
name_output_wide = ['ACS5 ', indicator_name, ' Wide MSA.xlsx']

name_output_long = "".join(name_output_long)
name_output_wide = "".join(name_output_wide)


In [ ]:
# Export long
with pd.ExcelWriter(os.path.join(path_out, sp_folder_out, name_output_long), engine='xlsxwriter') as writer:
    df_acs      .to_excel(writer, index = False, sheet_name = 'Full MSA')
    df_msa1     .to_excel(writer, index = False, sheet_name = 'MSA'     )


In [ ]:
# Export wide
with pd.ExcelWriter(os.path.join(path_out, sp_folder_out, name_output_wide), engine='xlsxwriter') as writer:
    df_msa2_pop      .to_excel(writer, index = False, sheet_name = 'MSA Total')
    df_msa2_prop     .to_excel(writer, index = False, sheet_name = 'MSA Prop' )
